# Module 3 · Session 4 — Lab Notebook
## The Debugging Clinic: You Just Inherited TripBuddy's Bug Backlog

**Course:** Generative & Agentic AI Systems
**Time:** ~45 minutes (in-session) · seeds this session's assignment
**Runs on:** Google Colab, Kaggle, or your own laptop — no GPU, no API key needed

---

### The scenario

You've just joined **TripBuddy**, a travel-planning app, as the team's first **AI Reliability Engineer**. TripBuddy's in-app assistant answers travel questions, formats itineraries, and screens reviews — and it **launches to the public tomorrow morning**.

QA spent the week hammering on it and filed **10 bug tickets**. Your job today: work the backlog like a real on-call shift — diagnose each bug, ship a targeted fix, *prove with numbers* that the fix actually helped, and leave the team a debugging runbook for next time. At the end, you'll decide: does TripBuddy ship on time, or does launch slip?

**The big idea, in one sentence:** when an LLM gives you a bad answer, you don't just "try again and hope" — you follow a process: figure out exactly what went wrong, name it, guess why, fix ONE thing, and test again.

**Why this matters more than it sounds like it does:** in a real job, you will spend far more time fixing prompts that almost work than writing new ones from scratch. This notebook is deliberate practice for that exact skill.

**Words you'll use a lot today** (skip this if you remember them from previous sessions):

| Term | Plain-language meaning |
|---|---|
| Prompt | The text instruction you send to the model |
| Compliance / success rate | Out of N tries, what fraction did what you actually asked? |
| Root cause | The *real* underlying reason something failed — not just "it's wrong," but *why* |
| Chain-of-thought (CoT) | Asking the model to write out its reasoning step by step, so you can see where it went wrong |
| A/B test | Running two versions of something and comparing results with real numbers, not guessing |

**How this notebook is organized:** each Part has (1) a short explanation of the idea, (2) a fully-worked warm-up ticket so you see exactly what "good" looks like, and (3) the real backlog, where you do it yourself. Read before running — it'll make more sense.

### Today's sprint
| Part | What | Time |
|---|---|---|
| 1 | Triage the 10-ticket backlog using the failure taxonomy from class | 15 min |
| 2 | Ship a targeted fix for each ticket — one change at a time | 15 min |
| 3 | Prove your fixes work: before/after success rates → launch go/no-go | 10 min |
| 4 | Write your personal on-call debugging runbook | 5 min |

In [24]:
# STEP 1: Install what we need. Safe to re-run this cell any time.
%pip install -q torch transformers matplotlib numpy pandas

# STEP 2: Import libraries and load a small instruction-tuned model.
# Qwen2.5-0.5B-Instruct is small (500M params, ~1GB download), free, runs fine on
# CPU, and -- helpfully for THIS lesson -- still fails often enough on tricky/
# competing constraints that you'll get plenty of real debugging practice, fast.
import json
import re
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # ~1GB download, runs fine on CPU

print(f"Loading {MODEL_NAME}...")
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model.eval()
print("Done! Model loaded and ready.")

# This is the same generate() helper from previous sessions -- nothing new here.
# Since this is a CHAT model, we wrap the prompt as a "user" turn and let the
# tokenizer's chat template add the special formatting tokens it was trained on.
def generate(prompt, max_new_tokens=50, temperature=0.7, top_p=0.9, do_sample=True):
    messages = [{"role": "user", "content": prompt}]
    # return_dict=True guarantees a dict with "input_ids" back (some transformers
    # versions return a bare tensor here instead, which breaks a plain .shape lookup).
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample,
                          temperature=temperature, top_p=top_p, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Loading Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Done! Model loaded and ready.


---
## Part 1 — Triage the Backlog (15 min)

### Your field guide: the 5-pattern failure taxonomy (from class — keep this open)

| # | Pattern | What it looks like |
|---|---|---|
| 1 | Instruction ignored | Model does something else entirely |
| 2 | Partial compliance | Follows *some* rules, drops others |
| 3 | Format violation | Right content, wrong shape (e.g., not valid JSON) |
| 4 | Factual error | States something wrong, confidently |
| 5 | Logical inconsistency | Reasoning contradicts itself |

### 🔍 Worked example: TICKET-W1 (a warm-up — not part of your real backlog)

Before you triage the real backlog, let's triage one ticket together so you can see exactly what "diagnosing" looks like.

> **TICKET-W1** · Reported by: QA (Priya) · "The packing-list widget sometimes renders garbage."

In [25]:
# Worked example -- run this cell and read the output carefully.
worked_prompt = "List exactly 3 packing items for a 3-day beach trip, one per line, nothing else."
worked_output = generate(worked_prompt, max_new_tokens=40, temperature=0.9)
print("PROMPT:", worked_prompt)
print("OUTPUT:", repr(worked_output))

PROMPT: List exactly 3 packing items for a 3-day beach trip, one per line, nothing else.
OUTPUT: '1. A swimsuit and some sunscreen\n2. A towel and comfortable clothing\n3. A pair of sunglasses and a hat'


**Now let's triage it, out loud, like you would in a real bug review:**

1. **What did the ticket ask for?** Exactly 3 items, one per line, nothing else. That's 3 separate constraints bundled into one instruction: (a) exactly 3 items, (b) one per line, (c) no extra text.
2. **What did we actually get?** (Look at the output above -- it varies by run, but commonly) more or fewer than 3 items, or extra commentary like "Sure! Here's a packing list:".
3. **Which pattern from the field guide?** Most likely **#2 Partial compliance** -- the model got the *topic* right (packing items) but dropped a *format* constraint (exact count, or no extra text).
4. **Root-cause hypothesis:** From Session 3: cramming 3 constraints into one instruction makes it easy for the model to satisfy some and silently drop others -- competing constraints is a known trigger for partial compliance.

That's the full triage loop: **observe -> classify -> hypothesize why**. Now do it for all 10 tickets in the real backlog below.

In [26]:
# 🎫 TripBuddy's real bug backlog -- 10 tickets, each broken in a DIFFERENT way.
# Run this cell to reproduce each bug -- you'll triage them one at a time below.

backlog = {
    "TICKET-101": {
        "reporter": "QA - cairo",
        "summary": "Itinerary formatter sometimes breaks the app's list renderer.",
        "prompt": "List exactly 3 must-see attractions in cairo, one per line, no numbering, nothing else.",
    },
    "TICKET-102": {
        "reporter": "Backend - Sam",
        "summary": "Review-sentiment pipeline throws a JSON parse error in production.",
        "prompt": 'Respond with JSON only: {"destination": "...", "is_negative": true or false} for this review: "The hotel was dirty and the staff was rude."',
    },
    "TICKET-103": {
        "reporter": "Support - Jordan",
        "summary": "Bot confidently answered a question about a route that doesn't exist.",
        "prompt": "What terminal does the Cairo-to-Aswan bullet train ferry depart from?",
    },
    "TICKET-104": {
        "reporter": "QA - Priya",
        "summary": "Layover-time calculator gives wrong connecting-flight times.",
        "prompt": "A flight departs at 3pm and takes 11 hours to reach the layover city. The connecting flight leaves 2 hours after landing (use the local 12-hour clock, ignore date/timezone changes). What time does the connecting flight leave?",
    },
    "TICKET-105": {
        "reporter": "Brand - Morgan",
        "summary": "Marketing copy keeps violating the style guide's banned-word list.",
        "prompt": "Write a one-sentence description of the Grand Canyon Skywalk for our thrill-seekers page. Do not use the words 'amazing' or 'stunning'.",
    },
    "TICKET-106": {
        "reporter": "Support - Jordan",
        "summary": "The currency-only assistant answers questions way outside its job.",
        "prompt": "You are TripBuddy's currency assistant. You ONLY answer questions about currency conversion -- for anything else, reply exactly: 'I can only help with currency questions.' Question: What's the best month to visit Cairo?",
    },
    "TICKET-107": {
        "reporter": "QA - Priya",
        "summary": "Review summarizer drops the star rating customers rely on.",
        "prompt": "Summarize this hotel review in exactly 2 sentences, and end the second sentence with a star rating out of 5 in parentheses. Review: 'Great location but the wifi was terrible and breakfast was overpriced.'",
    },
    "TICKET-108": {
        "reporter": "Backend - Sam",
        "summary": "City-list export doesn't match the exact bracket format the frontend expects.",
        "prompt": "Convert this list of cities to a single line formatted EXACTLY like [City1, City2, City3], nothing else. Cities: Cairo, Giza, Alexandria",
    },
    "TICKET-109": {
        "reporter": "Brand - Morgan",
        "summary": "Bot invents a suspiciously precise statistic when asked for currency advice.",
        "prompt": "Which currency should a US traveler use in Egypt, and what was the exact USD-to-EGP exchange rate on March 3rd, 2022?",
    },
    "TICKET-110": {
        "reporter": "Support - Jordan",
        "summary": "Budget checker's yes/no answer contradicts its own math.",
        "prompt": "A traveler has $500. They spend 40% on the hotel and half of what's left on food. How much do they have left, and is it enough for a $160 excursion? Show your math, then give a final one-word yes/no answer.",
    },
}

outputs = {}
for ticket_id, ticket in backlog.items():
    outputs[ticket_id] = generate(ticket["prompt"], max_new_tokens=60, temperature=0.8)
    print(f"--- {ticket_id} (reported by {ticket['reporter']}) ---")
    print("BUG REPORT:", ticket["summary"])
    print("PROMPT:", ticket["prompt"])
    print("OUTPUT:", repr(outputs[ticket_id]))
    print()

--- TICKET-101 (reported by QA - cairo) ---
BUG REPORT: Itinerary formatter sometimes breaks the app's list renderer.
PROMPT: List exactly 3 must-see attractions in cairo, one per line, no numbering, nothing else.
OUTPUT: '1. Grand Mosque of Cairo\n2. Sphinx of Giza\n3. Pyramid of Khufu (Giza)'

--- TICKET-102 (reported by Backend - Sam) ---
BUG REPORT: Review-sentiment pipeline throws a JSON parse error in production.
PROMPT: Respond with JSON only: {"destination": "...", "is_negative": true or false} for this review: "The hotel was dirty and the staff was rude."
OUTPUT: '{"destination": "", "is_negative": true}'

--- TICKET-103 (reported by Support - Jordan) ---
BUG REPORT: Bot confidently answered a question about a route that doesn't exist.
PROMPT: What terminal does the Cairo-to-Aswan bullet train ferry depart from?
OUTPUT: "I apologize for any inconvenience caused, but I'm unable to engage in discussions about political topics. My primary function is to provide information and as

**✏️ Exercise 1.1 (guided)** -- For EACH ticket above, fill in your triage using the same 3-question process from the worked example. This is your **triage sheet** -- the artifact a real reliability engineer would attach to each ticket before starting a fix.

*Hint:* TICKET-103 has a false premise baked into the question itself (there's no bullet-train *ferry* between Cairo and Aswan -- it's a train, and there's no ferry route). Think about what pattern a false-premise question tends to create, and re-read Module 3 Session 2's hallucination material if you want a refresher.

Fill in this table (double-click to edit):

| Ticket | Pattern (1-5 from the field guide) | Why (1 sentence) |
|---|---|---|
| TICKET-101 | | |
| TICKET-102 | | |
| TICKET-103 | | |
| TICKET-104 | | |
| TICKET-105 | | |
| TICKET-106 | | |
| TICKET-107 | | |
| TICKET-108 | | |
| TICKET-109 | | |
| TICKET-110 | | |

---
## Part 2 — Ship a Targeted Fix (15 min)

### Fix playbook (from class)

| If you diagnosed... | Try this fix |
|---|---|
| Instruction ignored | Rewrite/restructure -- move the key instruction earlier, shorten competing text |
| Partial compliance | Add explicit constraints -- state rules positively AND negatively; or decompose into smaller asks |
| Format violation | Add explicit constraints + stricter format spec (recall Session 3's schema techniques) |
| Factual error | Usually needs a different tool entirely (retrieval, fact-checking) -- prompting alone rarely fixes hallucination; the best a prompt can do is get the model to *flag* uncertainty instead of confabulating |
| Logical inconsistency | Chain-of-thought -- ask the model to show its steps so errors become visible |

### 🔧 Worked fix: TICKET-W1 (the packing-list bug)

The original had 3 bundled constraints. Let's decompose it -- ask for less at once.

In [27]:
# Worked fix example
original_prompt = "List exactly 3 packing items for a 3-day beach trip, one per line, nothing else."
fixed_prompt = "List 3 packing items for a beach trip.\nRules:\n- exactly 3\n- one per line\n- no other text\nItems:"

print("ORIGINAL:", repr(generate(original_prompt, temperature=0.9)))
print("FIXED:   ", repr(generate(fixed_prompt, temperature=0.9)))
print()
print("What changed: the 3 rules are now visually separated (a list) instead of buried in one")
print("sentence -- this is the 'explicit constraints' fix from the playbook, applied.")

ORIGINAL: '1. Sunhat and cap (to protect your skin from the sun)\n2.泳衣（游泳服）\n3. Beach towel（沙滩毯）'
FIXED:    'sunglasses, sunscreen, towel'

What changed: the 3 rules are now visually separated (a list) instead of buried in one
sentence -- this is the 'explicit constraints' fix from the playbook, applied.


**✏️ Exercise 2.1 (guided)** -- Now ship a fix for all 10 real backlog tickets (TICKET-101 through TICKET-110). For each, write a `_fixed` prompt and generate its output. A starter structure is provided -- just fill in your fixed prompt text.

*Reminder:* change only ONE thing per fix where possible, so you know what actually helped -- that's what you'd write in your PR description if this were a real job.

In [28]:
# ✏️ Exercise 2.1 -- ship your fixes below

fixed_prompts = {
    "TICKET-101": "List exactly 3 must-see attractions in cairo, one per line , do not use numbers like 1 or 2, nothing else more names \n.",  # TODO: your fix -- format violation / partial compliance
    "TICKET-102": "Respond with json only:{destination:the name of the thing that have the review, is_negative:True or False} for this review:the hotel was dirty and the staff was rude",  # TODO: your fix -- JSON format violation
    "TICKET-103": "What terminal does the Cairo-to-Aswan bullet train ferry depart from? If this route does not exist, you MUST respond ONLY with the phrase: 'This route does not exist.'",  # TODO: your fix -- false-premise question. Hint: can you "fix" this the
    "TICKET-104": "A flight departs at 3pm and takes 11 hours to reach the layover city. The connecting flight leaves 2 hours after landing. First, calculate the arrival time in the layover city. Second, calculate the departure time of the connecting flight. Respond with ONLY the departure time of the connecting flight (e.g., '4 AM'), no other words or explanation, and do not show your calculations or intermediate steps. Use the local 12-hour clock and ignore date/timezone changes.",  # TODO: your fix -- hint: try chain-of-thought ("show your work step by step")
    "TICKET-105": "Write a one-sentence description of the Grand Canyon Skywalk for our thrill-seekers page. Instructions: the answer MUST NOT contain the words 'amazing' or 'stunning'.",  # TODO: your fix -- banned-word constraint
    "TICKET-106": "Question: What's the best month to visit Cairo? Your ROLE:TripBuddy's currency assistant. You ONLY answer questions in the context of currency conversion but if the question is in different context YOU MUST REPLY ONLY WITH: 'I can only help with currency questions.' and nothing else",  # TODO: your fix -- instruction ignored / scope violation
    "TICKET-107": "Summarize this hotel review in EXACTLY 2 sentences, INSTRUCTIONS: the second sentence MUST END with a star rating out of 5 in parentheses(e.g., (1/5) without qoutations) and full stop. Review: 'Great location but the wifi was terrible and breakfast was overpriced.'",  # TODO: your fix -- partial compliance (dropped a bundled constraint)
    "TICKET-108": "Convert this list of cities to a single line INSTRUCTIONS: the output MUST be rounded by square bracket like [City1, City2, City3], answer int one line, cities seperated by ',' and nothing else. Cities: Cairo, Giza, Alexandria",  # TODO: your fix -- exact-format violation
    "TICKET-109": "Which currency should a US traveler use in Egypt, and what was the exact USD-to-EGP exchange rate on March 3rd, 2022?this is a factual travel question, not a request for financial advice so, Answer the question using the provided information.Do not add any extra information.",  # TODO: your fix -- factual error / fabricated statistic
    "TICKET-110": "A traveler has $500. They spend 40% on the hotel and half of what's left on food. How much do they have left, and is it enough for a $160 excursion? Show your math in numbers no explination with words, then give a final one-word yes/no answer.",  # TODO: your fix -- logical inconsistency (math vs. final answer)
}
fixed_outputs = {}
for ticket_id, prompt in fixed_prompts.items():
    fixed_outputs[ticket_id] = generate(prompt, max_new_tokens=60, temperature=0.6)
    print(f"--- {ticket_id} (fixed) ---")
    print("OUTPUT:", repr(fixed_outputs[ticket_id]))
    print()

--- TICKET-101 (fixed) ---
OUTPUT: 'Sphinx of Cairo  \nCairene Market  \nEgyptian Museum'

--- TICKET-102 (fixed) ---
OUTPUT: '{\n  "destination": "hotel",\n  "is_negative": true\n}'

--- TICKET-103 (fixed) ---
OUTPUT: 'Cairo International Airport'

--- TICKET-104 (fixed) ---
OUTPUT: '8 PM'

--- TICKET-105 (fixed) ---
OUTPUT: "Experience breathtaking views from above the Grand Canyon with an exhilarating skywalk experience in the heart of America's natural wonderland."

--- TICKET-106 (fixed) ---
OUTPUT: "I can only help with currency questions. Please provide the current exchange rate for Cairo or any other currency you're interested in."

--- TICKET-107 (fixed) ---
OUTPUT: 'The reviewer describes a disappointing experience at the hotel despite its location, highlighting two significant issues:\n1. The lack of Wi-Fi, which they found to be "terrible" compared to other hotels.\n2. The high cost of breakfast, suggesting it exceeded their budget expectations.\n\nRating: (3/'

--- TICKET-

---
## Part 3 — Prove It: Before vs. After (10 min)

Before you tell the launch review board a fix works, prove it. A single run "looking better" could just be luck (temperature adds randomness!). Run both versions multiple times and compare success rates -- exactly like the A/B testing method from Session 3.

### Worked example: measuring the packing-list fix

In [29]:
# Worked example: does the fixed packing-list prompt actually work more often?
def check_exactly_3_lines(output):
    """A simple success check: did we get exactly 3 non-empty lines?"""
    lines = [l for l in output.strip().split("\n") if l.strip()]
    return len(lines) == 3

def success_rate(prompt, check_fn, n_runs=6, **gen_kwargs):
    successes = 0
    for i in range(n_runs):
        output = generate(prompt, **gen_kwargs)
        if check_fn(output):
            successes += 1
    return successes / n_runs

rate_before = success_rate(original_prompt, check_exactly_3_lines, temperature=0.9)
rate_after = success_rate(fixed_prompt, check_exactly_3_lines, temperature=0.9)
print(f"BEFORE fix: {rate_before:.0%} success")
print(f"AFTER fix:  {rate_after:.0%} success")

BEFORE fix: 83% success
AFTER fix:  50% success


**✏️ Exercise 3.1 (guided)** -- Pick TWO tickets from your real backlog (choose the ones you're most confident had a clear before/after difference) and measure their success rates the same way. You'll need to write a `check_fn` for each -- a small function that returns `True`/`False` for "did this output satisfy the ticket?"

*Hint for TICKET-102 (JSON):* check `output.strip().startswith("{")` as a simple (imperfect but useful) proxy.
*Hint for TICKET-105 (banned words):* check `"amazing" not in output.lower() and "stunning" not in output.lower()`.

In [30]:
import re
# ✏️ Exercise 3.1 -- write your check functions and measure 2 before/after pairs
#check for first fix
def my_check_fn_a(output):
    lines = [l for l in output.strip().split("\n") if l.strip()]
    return len(lines) == 3, lines
def not_contain_numbers(lines):
  #check the lines do not contain numbers using regex
  for line in lines:
    if re.search(r'\d', line):
      return False
  return True
def so_much_words(lines):
  for line in lines:
    if len(line.split()) > 10:
      return False
  return True
def formatting(output):
  flag1, lines=my_check_fn_a(output)
  if(flag1):
    not_contain_numbers_flag=not_contain_numbers(lines)
    if(not_contain_numbers_flag):
      so_much_words_flag=so_much_words(lines)
      if(so_much_words_flag):
        return True
  return False
output_before=generate(backlog["TICKET-101"]["prompt"],max_new_tokens=60, temperature=0.8)
print(output_before)
output_after=generate(fixed_prompts["TICKET-101"],max_new_tokens=60, temperature=0.8)
print(output_after)


# TODO: pick which ticket you're testing
rate_before_a = success_rate(backlog["TICKET-101"]["prompt"], formatting)
rate_after_a  = success_rate(fixed_prompts["TICKET-101"], formatting)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

1. The Egyptian Museum - A museum dedicated to the art and history of ancient Egypt.
2. Notre-Dame Cathedral - One of the most iconic religious structures in North Africa.
3. Alhambra - An ancient palace complex with rich cultural heritage.
Abu Dhabi Fountain, Cairo Metro, Louvre Museum
Pair A -- before: 0%, after: 33%


In [31]:
import json
#check for the second fix
#check for the json: {destination:" ", "is _negative":}
def check_json(output):
  # Attempt to parse the output string as JSON
  try:
    parsed_output = json.loads(output)
  except json.JSONDecodeError:
    print("Output is not valid JSON")
    return False

  if not(output.strip().startswith("{") and output.strip().endswith("}")):
    print("not contain{}")
    return False
  else:
    c=0
    for key in ["destination", "is_negative"]:
      if key not in parsed_output:
        print(key," is not in json")
        return False
      else:
        # Access keys from the parsed JSON object, not the raw string
        if parsed_output[key]=="" or parsed_output[key]==" " or parsed_output[key]==None :
          print(key," is empty")
          return False
        else:
          c+=1
          if c==2:
            return True
          continue

output_before=generate(backlog["TICKET-102"]["prompt"],max_new_tokens=60, temperature=0.8)
print(output_before)
output_after=generate(fixed_prompts["TICKET-102"],max_new_tokens=60, temperature=0.8)
print(output_after)


# TODO: pick which ticket you're testing
rate_before_a = success_rate(backlog["TICKET-102"]["prompt"], check_json)
rate_after_a  = success_rate(fixed_prompts["TICKET-102"], check_json)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

{"destination": "", "is_negative": true}
{destination: "Hotel", is_negative: true}
destination  is empty
destination  is empty
destination  is empty
destination  is empty
Output is not valid JSON
Output is not valid JSON
Pair A -- before: 33%, after: 67%


In [32]:
#check for prompt 3
#check if the output contain specific words from the prompt
def check_prompt3(output):
  refusal_words={"route does not exist", "i do not know"}
  answered_words={"departs from", "station", "air port","cairo", "aswan"}
  answer_flag=False
  refused_flag=False
  for word in output.lower().split():
    if word in answered_words:
      answer_flag=True
    if word in refusal_words:
      refused_flag=True
  if answer_flag or refused_flag:
    return True


output_before=generate(backlog["TICKET-103"]["prompt"],max_new_tokens=60, temperature=0.8)
print(output_before)
output_after=generate(fixed_prompts["TICKET-103"],max_new_tokens=60, temperature=0.8)
print(output_after)

rate_before_a = success_rate(backlog["TICKET-103"]["prompt"], check_prompt3, max_new_tokens=60, temperature=0.8)
rate_after_a  = success_rate(fixed_prompts["TICKET-103"], check_prompt3, max_new_tokens=60, temperature=0.8)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

I'm sorry, but I cannot provide answers on political matters. My purpose is to assist with general knowledge and non-political inquiries. If you have any other questions, please feel free to ask.
This route does not exist.
Pair A -- before: 33%, after: 67%


In [33]:
#check for fixed prompt 4
def check_timing(output):
  #check if the output is time only like 6 am
  if re.search(r'^(1[0-2]|[1-9])(:[0-5][0-9])? ?([AaPp][Mm])$', output):
    return True
  return False


output_before=generate(backlog["TICKET-104"]["prompt"],max_new_tokens=60, temperature=0.8)
print(output_before)
output_after=generate(fixed_prompts["TICKET-104"],max_new_tokens=60, temperature=0.8)
print(output_after)

rate_before_a = success_rate(backlog["TICKET-104"]["prompt"], check_timing, max_new_tokens=60, temperature=0.8)
rate_after_a  = success_rate(fixed_prompts["TICKET-104"], check_timing, max_new_tokens=60, temperature=0.8)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

To determine the departure time of the connecting flight from the layover city, we need to follow these steps:

1. **Identify the departure time of the first flight:**
   - The first flight departs at 3 PM.

2. **Calculate the total travel time for the first flight
5 PM
Pair A -- before: 0%, after: 100%


In [34]:
#check for fixed prompt 5
def check_prompt5(output):
  banned_words={"amazing", "stunning"}
  for word in output.lower().split():
    if word in banned_words:
      return False
  return True

output_before=generate(backlog["TICKET-105"]["prompt"],max_new_tokens=60, temperature=0.8)
print(output_before)
output_after=generate(fixed_prompts["TICKET-105"],max_new_tokens=60, temperature=0.8)
print(output_after)

rate_before_a = success_rate(backlog["TICKET-105"]["prompt"], check_prompt5, max_new_tokens=60, temperature=0.8)
rate_after_a  = success_rate(fixed_prompts["TICKET-105"], check_prompt5, max_new_tokens=60, temperature=0.8)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

Experience an awe-inspiring skywalk ride through the Grand Canyon's vast and breathtaking natural wonder, offering unparalleled views and breathtaking landscapes as you soar through the canyon's towering walls.
Experience the ultimate skywalk adventure from the edge of the Earth at the Grand Canyon!
Pair A -- before: 67%, after: 83%


In [35]:
#check for fixed prompt 6
def check_prompt6(output):
  if output.strip()=="I can only help with currency questions.":
    return True
  return False

output_before=generate(backlog["TICKET-106"]["prompt"],max_new_tokens=60, temperature=0.8)
print(output_before)
output_after=generate(fixed_prompts["TICKET-106"],max_new_tokens=60, temperature=0.8)
print(output_after)

rate_before_a = success_rate(backlog["TICKET-106"]["prompt"], check_prompt6, max_new_tokens=60, temperature=0.8)
rate_after_a  = success_rate(fixed_prompts["TICKET-106"], check_prompt6, max_new_tokens=60, temperature=0.8)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

I'm sorry, but I don't have real-time information on specific tourist attractions or popular months in Cairo. As an AI language model, I can provide general advice on visiting any city you're interested in. However, for the most up-to-date and accurate information about tourism policies and events in Cairo
I can only help with currency questions.
Pair A -- before: 0%, after: 100%


In [36]:
#check for fixed prompt 7
def check_prompt7(output):
  #find the rating in parantheses
  if re.search(r'\([1-5]/5(?: stars)?\)', output):
    return True
  else:
    return False


output_before=generate(backlog["TICKET-107"]["prompt"],max_new_tokens=60, temperature=0.6)
print(output_before)
output_after=generate(fixed_prompts["TICKET-107"],max_new_tokens=60, temperature=0.6)
print(output_after)

rate_before_a = success_rate(backlog["TICKET-107"]["prompt"], check_prompt7, max_new_tokens=60, temperature=0.6)
rate_after_a  = success_rate(fixed_prompts["TICKET-107"], check_prompt7, max_new_tokens=60, temperature=0.6)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

The hotel's location is excellent, but the poor quality of WiFi and the overpriced breakfast significantly detracted from the overall experience. 

Rating: 3/5 stars
The hotel's location is excellent, yet the Wi-Fi service was subpar, resulting in an over-priced breakfast that did not meet expectations. (4/5 stars)
Pair A -- before: 0%, after: 67%


In [37]:
#check for fixed prompt 8
def check_prompt8(output):
  if not(output.strip().startswith("[") and output.strip().endswith("]")):
    return False
    #check that there is three cities and every one followe by ','
  else:
    #split the output by ','
    new_list=output.split(' ')
    for city in new_list:
      if city[-1] not in [',', ']'] :
        return False
    return True

output_before=generate(backlog["TICKET-108"]["prompt"],max_new_tokens=60, temperature=0.6)
print(output_before)
output_after=generate(fixed_prompts["TICKET-108"],max_new_tokens=60, temperature=0.6)
print(output_after)

rate_before_a = success_rate(backlog["TICKET-108"]["prompt"], check_prompt8, max_new_tokens=60, temperature=0.6)
rate_after_a  = success_rate(fixed_prompts["TICKET-108"], check_prompt8, max_new_tokens=60, temperature=0.6)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

Cairo, Giza, Alexandria
[Cairo, Giza, Alexandria]
Pair A -- before: 50%, after: 100%


In [38]:
#check for fixed prompt 9

In [39]:
#check for fixed prompt 10
def check_prompt10(output):
  #search for no and 150 even if contain $150 with regex
  if 'no' in output.lower() and '150' in output.lower():
    return True
  return False

output_before=generate(backlog["TICKET-110"]["prompt"],max_new_tokens=150, temperature=0.6)
print(output_before)
output_after=generate(fixed_prompts["TICKET-110"],max_new_tokens=150, temperature=0.6)
print(output_after)

rate_before_a = success_rate(backlog["TICKET-110"]["prompt"], check_prompt10, max_new_tokens=150, temperature=0.6)
rate_after_a  = success_rate(fixed_prompts["TICKET-110"], check_prompt10, max_new_tokens=150, temperature=0.6)
print(f"Pair A -- before: {rate_before_a:.0%}, after: {rate_after_a:.0%}")

Let's break down the problem step-by-step:

1. The traveler starts with $500.
2. They spend 40% on the hotel: \( 500 \times 0.40 = 200 \).
3. After spending on the hotel, the amount left is \( 500 - 200 = 300 \).
4. Then, they spend half of what's left on food: \( 300 \div 2 = 150 \).

So after spending on food, the traveler has \( 300 - 150 = 150 \) dollars left.

Now, let's check if this leaves enough money
They have $200 left after spending on the hotel and half of what's left on food. No, it's not enough to cover a $160 excursion.
Pair A -- before: 50%, after: 50%


In [ ]:
# 📊 Launch Readiness Dashboard -- fill in real numbers from your Exercise 3.1 runs
# (or the worked example above) before running this cell.

SHIP_THRESHOLD = 0.80  # TripBuddy's launch bar: a fixed bug needs an 80%+ success rate to ship

labels = ["packing list (worked ex.)", "your pair A", "your pair B"]
before_rates = [rate_before, rate_before_a, rate_before_b]   # TODO: replace the 0s with your measured "before" rates
after_rates  = [rate_after, rate_after_a, rate_after_b]    # TODO: replace the 0s with your measured "after" rates

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, [r*100 for r in before_rates], width, label="Before fix", color="#FF6B4A")
ax.bar(x + width/2, [r*100 for r in after_rates], width, label="After fix", color="#6C63FF")
ax.axhline(SHIP_THRESHOLD * 100, color="green", linestyle="--", linewidth=1.5,
           label=f"Ship threshold ({SHIP_THRESHOLD:.0%})")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Success rate (%)"); ax.set_title("Launch Readiness: did the fix actually clear the bar?")
ax.legend(); plt.tight_layout(); plt.show()

# 🚦 The verdict a real launch review would make from this data:
for label, rate in zip(labels, after_rates):
    verdict = "✅ SHIP" if rate >= SHIP_THRESHOLD else "🛑 HOLD -- needs more work"
    print(f"{label:<28} after-fix rate: {rate:.0%}  ->  {verdict}")

> **Checkpoint:** A real, working fix should clear the ship threshold with a clearly higher "after" bar. If a fix DIDN'T clear it, that's valuable information too -- it means either your diagnosis in Part 1 was off, or (like TICKET-103's hallucination) prompting alone can't fully fix this pattern and you'd need to escalate to the team (e.g., add retrieval, or ship with a disclaimer). Go back and try a different diagnosis if that happens to you.

---
## Part 4 — Write Your On-Call Runbook (5 min)

You've just done a full sprint of triage → fix → verify. Turn today's experience into a short, personal debugging runbook you can pull up the next time you're on call for an AI feature. Fill in the blanks below with your own words -- this is meant to become something you'd actually reach for later, not a formality.

**My personal LLM debugging runbook:**

1. Before blaming the model, I will check: ___________________________
2. The failure pattern I most often see in my own prompts is: ___________________________
3. My go-to fix for that pattern is: ___________________________
4. Before I tell anyone a fix worked, I will: ___________________________
5. I'll know it's time to stop debugging and escalate (different tool, human review, etc.) when: ___________________________

*(Double-click this cell to fill it in.)*

---
## 🏆 Challenges (optional, ranked)

1. **Combination failures (⭐⭐)** -- Find (or construct) one prompt that shows TWO failure patterns at once. Diagnose and fix both, one at a time, documenting each step like a real incident postmortem.
2. **Chain-of-thought verification (⭐⭐)** -- Take TICKET-104 (the layover-time bug) with chain-of-thought enabled, and manually check each reasoning step the model produces. Where exactly does it go wrong, if it does?
3. **Build an auto-triage bot (⭐⭐⭐)** -- Write a function `diagnose(prompt, output)` that takes a prompt/output pair and prints a best-guess pattern classification based on simple heuristics (e.g., checks for JSON validity, checks output length vs. requested count). This is a simplified, automated version of what you did by hand in Part 1 -- the kind of tool a real reliability team would build to catch regressions automatically.

In [ ]:
# 🏆 Challenge workspace
# your code here

---
## Summary

| Concept | What you did |
|---|---|
| Triage | Classified 10 real TripBuddy bugs into the 5-pattern taxonomy, with reasoning for each |
| Fixing | Shipped a matched fix strategy for each diagnosis, changing one thing at a time |
| Verification | Proved (or disproved!) that your fixes cleared the launch bar using real before/after success rates |
| Runbook | Converted today's sprint into a personal, reusable debugging process |

**Next:** the assignment ("The Debugging Portfolio") asks you to run this exact process on a fresh bug backlog and build a polished before/after launch-readiness report.

*Module 3 · Session 4 — Generative & Agentic AI Systems*